In [14]:
import $ivy.`org.apache.spark::spark-sql:3.4.1`
import $ivy.`org.apache.spark::spark-mllib:3.4.1`
import $ivy.`org.scalanlp::breeze:2.1.0`

import breeze.linalg._
import breeze.numerics._
import scala.util.Random
import org.apache.spark.sql.SparkSession

import $ivy.$
import $ivy.$
import $ivy.$
import breeze.linalg._
import breeze.numerics._
import scala.util.Random
import org.apache.spark.sql.SparkSession

In [15]:
val spark = SparkSession.builder()
  .appName("Spark + Breeze Test")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext
println("Spark session initialized")

Spark session initialized


spark: SparkSession = org.apache.spark.sql.SparkSession@1e90d871
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@1a4434b0

In [16]:
val numPoints = 100000
val rnd = new Random(42)

// Hidden model
val trueWeights = DenseVector(1.5, 0.3, -0.7)

// Feature matrix: 100000 rows × 3 features
val data = DenseMatrix.rand[Double](numPoints, 3)

// Noise generation
val noise = DenseVector(Array.fill(numPoints)(rnd.nextGaussian() * 0.1))

// Labels: X * w + noise
val labels = data * trueWeights + noise


numPoints: Int = 100000
rnd: Random = scala.util.Random@4d61066c
trueWeights: DenseVector[Double] = DenseVector(1.5, 0.3, -0.7)
data: DenseMatrix[Double] = 0.5138250687282788    0.004168872719445549  0.8233399013531759     
0.5876174345291456    0.7871180710995731    0.9733353732672476     
0.2518621694098637    0.3511808635321163    0.3516666723057815     
0.24203945638873225   0.5004242247482642    0.32622573918445186    
0.4152597288770854    0.014789460453357162  0.5891818854596651     
0.6336290412737873    0.7181555285750452    0.9351697667533991     
0.14430718458486957   0.705969289417937     0.2938470637439998     
0.13022152767338424   0.38936841871289873   0.9093215830230594     
0.764067072698438     0.22216751861845507   0.07547036900170423    
0.4684447271907297    0.6365876951822431    0.5113192376233084     
0.13970915031303144   0.024509658374932686  0.21035884359928403    
0.7423623799032335    0.9014460508469131    0.03428241606305016    
0.4850400743627321    0.3648

In [17]:
def gradientDescent(
  data: DenseMatrix[Double],
  labels: DenseVector[Double],
  learningRate: Double,
  numIterations: Int
): DenseVector[Double] = {
  val numFeatures = data.cols
  var weights = DenseVector.zeros[Double](numFeatures)

  for (i <- 0 until numIterations) {
    val predictions = data * weights
    val errors = predictions - labels
    val gradient = (data.t * errors) / data.rows.toDouble
    weights -= learningRate * gradient

    if (i % 100 == 0) {
      val loss = sum(pow(predictions - labels, 2.0)) / data.rows.toDouble
      println(s"Iteration $i: Loss = $loss")
    }
  }

  weights
}

val weights = gradientDescent(data, labels, learningRate = 0.001, numIterations = 1000)
println(s"Final weights: $weights")

Iteration 0: Loss = 0.5465913089662813
Iteration 100: Loss = 0.49188218587008103
Iteration 200: Loss = 0.44512057943549954
Iteration 300: Loss = 0.40509317137166156
Iteration 400: Loss = 0.3707729001047134
Iteration 500: Loss = 0.3412903515087545
Iteration 600: Loss = 0.31590954433655105
Iteration 700: Loss = 0.29400743527100504
Iteration 800: Loss = 0.27505657221863106
Iteration 900: Loss = 0.25861041223726955
Final weights: DenseVector(0.2972963687441837, 0.20126820411086235, 0.1222680024936006)


defined function gradientDescent
weights: DenseVector[Double] = DenseVector(0.2972963687441837, 0.20126820411086235, 0.1222680024936006)

In [21]:
import org.apache.spark.ml.linalg.Vectors
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.sql.SparkSession
import spark.implicits._

// Assume that `data` and `labels` are already defined as arrays or collections

// Convert Breeze matrix to a DataFrame for Spark with explicit type specification
val df = (0 until numPoints).map { i =>
  val features = Vectors.dense(data(i, 0), data(i, 1), data(i, 2))
  val label = labels(i)
  (features, label)
}.toDF("features", "label").cache() 

// Create and customize a linear regression model
val lr = new LinearRegression()
  .setMaxIter(100)
  .setRegParam(0.0)
  .setElasticNetParam(0.0)

// Training the model
val model = lr.fit(df)

// Output coefficients and intercept
println(s"Coefficients from Spark ML: ${model.coefficients}")
println(s"Intercept from Spark ML: ${model.intercept}")

25/05/02 14:24:30 INFO BlockManagerInfo: Removed broadcast_7_piece0 on 6a9bd1833c0b:39523 in memory (size: 8.4 KiB, free: 2.2 GiB)
25/05/02 14:24:30 INFO Instrumentation: [d4c7ddb5] Stage class: LinearRegression
25/05/02 14:24:30 INFO Instrumentation: [d4c7ddb5] Stage uid: linReg_1baf1f217d38
25/05/02 14:24:30 INFO Instrumentation: [d4c7ddb5] training: numPartitions=12 storageLevel=StorageLevel(1 replicas)
25/05/02 14:24:30 INFO Instrumentation: [d4c7ddb5] {"elasticNetParam":0.0,"maxIter":100,"regParam":0.0}
25/05/02 14:24:30 INFO SparkContext: Starting job: head at DatasetUtils.scala:218
25/05/02 14:24:30 INFO DAGScheduler: Got job 4 (head at DatasetUtils.scala:218) with 1 output partitions
25/05/02 14:24:30 INFO DAGScheduler: Final stage: ResultStage 8 (head at DatasetUtils.scala:218)
25/05/02 14:24:30 INFO DAGScheduler: Parents of final stage: List()
25/05/02 14:24:30 INFO DAGScheduler: Missing parents: List()
25/05/02 14:24:30 INFO DAGScheduler: Submitting ResultStage 8 (MapPartiti

Coefficients from Spark ML: [1.50066330986285,0.3003295021769394,-0.7020009618914734]
Intercept from Spark ML: 4.828948644525085E-4


import org.apache.spark.ml.linalg.Vectors
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.sql.SparkSession
import spark.implicits._
df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [features: vector, label: double]
lr: LinearRegression = linReg_1baf1f217d38
model: org.apache.spark.ml.regression.LinearRegressionModel = LinearRegressionModel: uid=linReg_1baf1f217d38, numFeatures=3

In [20]:
println(s"True weights: $trueWeights")
println(s"Breeze weights: $weights")
println(s"Spark weights: ${model.coefficients.toArray.mkString(", ")}")

True weights: DenseVector(1.5, 0.3, -0.7)
Breeze weights: DenseVector(0.2972963687441837, 0.20126820411086235, 0.1222680024936006)
Spark weights: 1.5006633098628506, 0.30032950217693855, -0.7020009618914745
